# Crop Price Geographic Merging

This notebook performs spatial joins between crop price data and administrative boundary shapefiles to add district and chiefdom information to each price observation. It uses geographic coordinates (latitude/longitude) to map price data points to their corresponding administrative units.


## Setup and Data Loading

Initialize required libraries for geospatial operations and load both the administrative boundary shapefile and the combined crop price dataset. Set up file paths for input data and output location.


In [34]:
import geopandas as gpd
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from shapely.geometry import Point

# Base path. Change this to your dropbox folder
mypc_path = "/Users/justinblack/Library/CloudStorage/Dropbox"

# Data, output, input, and map folders. These should stay the same
data_folder = f'{mypc_path}/Lavoro_Replication/0_data_construction/'  
output_folder = f'{data_folder}/3_data_output'
output_file_name = "crop_prices_geo_mapped.dta"
output_path = f'{output_folder}/{output_file_name}'
input_folder = f'{data_folder}1_data_python/temp/'
map_folder = f'{data_folder}/4_output/py_maps/'

# Load the shapefile containing geographic boundaries
core_shape = gpd.read_file(f'{data_folder}1_data_python/admin_shapefile/geodf.shp')

# Load the crop price data
crop_price_df = pd.read_csv(f'{input_folder}combined_crop_prices.csv')

## Spatial Join and Geographic Merging

Perform the spatial join process to map crop price data to administrative boundaries:

1. **Data separation**: Split the dataset into records with and without geographic coordinates
2. **Coordinate processing**: Convert latitude/longitude coordinates to point geometries 
3. **Spatial join**: Use GeoPandas to perform a spatial join, matching price points to administrative polygons (districts and chiefdoms)
4. **Data recombination**: Merge the spatially-joined data back with records lacking coordinates
5. **Column standardization**: Ensure consistent column structure and remove duplicate/unnecessary fields

The result is a dataset where each price observation includes its corresponding district and chiefdom information when coordinates are available.


In [35]:
# Separate rows with and without coordinates
has_coords = crop_price_df.dropna(subset=['Latitude', 'Longitude']).copy()
no_coords = crop_price_df[crop_price_df[['Latitude', 'Longitude']].isna().any(axis=1)].copy()

# Process rows with coordinates through spatial join
if len(has_coords) > 0:
    # Convert to geometry
    geometry = gpd.points_from_xy(has_coords['Longitude'], has_coords['Latitude'])
    gdf = gpd.GeoDataFrame(has_coords, geometry=geometry, crs="EPSG:4326")  # assuming WGS84
    
    # Make sure core_shape is in same CRS
    core_shape = core_shape.to_crs("EPSG:4326")
    
    # Perform spatial join (points into polygons)
    joined_with_coords = gpd.sjoin(gdf, core_shape[['iso', 'district', 'chiefdom', 'geometry']], how="left", predicate="within")
    
    # Drop the join index column if added
    joined_with_coords = joined_with_coords.drop(columns=["index_right"], errors="ignore")
    
    # Drop all geometry columns
    joined_with_coords = joined_with_coords.drop(columns=joined_with_coords.select_dtypes(include='geometry').columns)
else:
    joined_with_coords = pd.DataFrame()

# Process rows without coordinates - add empty spatial columns
if len(no_coords) > 0:
    no_coords['district'] = None
    no_coords['chiefdom'] = None
    no_coords['iso'] = None  # This will be excluded later anyway
else:
    no_coords = pd.DataFrame()

# Combine both datasets
if len(joined_with_coords) > 0 and len(no_coords) > 0:
    # Ensure both dataframes have the same columns
    all_cols = set(joined_with_coords.columns) | set(no_coords.columns)
    for col in all_cols:
        if col not in joined_with_coords.columns:
            joined_with_coords[col] = None
        if col not in no_coords.columns:
            no_coords[col] = None
    
    # Combine the datasets
    joined = pd.concat([joined_with_coords, no_coords], ignore_index=True)
elif len(joined_with_coords) > 0:
    joined = joined_with_coords
elif len(no_coords) > 0:
    joined = no_coords
else:
    joined = pd.DataFrame()

# Reorder columns: put shapefile-derived names first
# Keep the original ISO column from crop data, exclude the duplicate iso from shapefile
first_cols = ["ISO", "district", "chiefdom"]

# Define columns to exclude from final dataframe
cols_to_exclude = ["Latitude", "Longitude", "index", "iso"]  # Exclude lowercase 'iso' from shapefile
other_cols = [col for col in joined.columns if col not in first_cols and col not in cols_to_exclude]
joined = joined[first_cols + other_cols]

# Export to Stata
joined.to_stata(output_path, write_index=False, version=117)

# Save locally (uncomment for debugging)
# joined.to_csv("crop_prices_geo_mapped.csv", index=False)

print(f"Successfully merged crop prices with geographic information and exported to Stata.")

Successfully merged crop prices with geographic information and exported to Stata.
